In [ ]:
from __future__ import annotations

from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import optax
import pandas as pd
import seaborn as sns
import torch
from numpy.polynomial.polynomial import Polynomial

from example_models.linear_chain import get_linear_chain_2v
from mxlpy import Model, Simulator, SurrogateProtocol, fns, npe, plot, scan, surrogates
from mxlpy.distributions import LogNormal, Normal, sample

# Seed the neural network training, so the examples are reproducible
_ = torch.manual_seed(0)


# Mechanistic Learning

Mechanistic learning is the intersection of mechanistic modelling and machine learning.  
*mxlpy* currently supports two such approaches: surrogates and neural posterior estimation.  

In the following we will mostly use the `mxlpy.surrogates` and `mxlpy.npe` modules to learn about both approaches.  

## Surrogate models


**Surrogate models** replace whole parts of a mechanistic model (or even the entire model) with machine learning models.  

<img src="assets/surrogate.png" style="max-height: 300px;">

This allows combining together multiple models of arbitrary size, without having to worry about the internal state of each model.  
They are especially useful for improving the description of *boundary effects*, e.g. a dynamic description of downstream consumption. 

## Manual construction

Surrogates can have return two kind of values in `mxply`: `derived quantities` and `reactions`.  

We will start by defining a polynomial surrogate that will get the value of a variable `x` and output the derived quantity `y`.  
Note that due to their nature surrogates can take multiple inputs and return multiple outputs, so we will always use iterables when defining them.  

We then also add a derived value `z` that uses the output of our surrogate to see that we are getting the correct output.   

In [ ]:
m = Model()
m.add_variable("x", 1.0)
m.add_surrogate(
    "surrogate",
    surrogates.poly.Surrogate(
        model=Polynomial(coef=[2]),
        args=["x"],
        outputs=["y"],
    ),
)
m.add_derived("z", fns.add, args=["x", "y"])

# Check output
m.get_args()

Next we extend that idea to create a reaction.  
The only thing we need to change here is to also add the `stoichiometries` of the respective output variable.  

I've renamed the output to `v1` here to fit convention, but that is not technically necessary.  
`mxlpy` will always infer structurally into what kind of value your surrogate will be translated.  


In [ ]:
m = Model()
m.add_variable("x", 1.0)
m.add_surrogate(
    "surrogate",
    surrogates.poly.Surrogate(
        model=Polynomial(coef=[2]),
        args=["x"],
        outputs=["v1"],
        stoichiometries={"v1": {"x": -1}},
    ),
)
m.add_derived("z", fns.add, args=["x", "v1"])

# Check output
m.get_right_hand_side()

Note that if you have **multiple outputs**, it is perfectly fine for them to mix between derived values and reactions.  

```python
Surrogate(
    model=...,
    args=["x", "y"],
    outputs=["d1", "v1"],               # outputs derived value d1 and rate v1
    stoichiometries={"v1": {"x": -1}},  # only rate v1 is given stoichiometries
)
```

## Training a surrogate from data and using it

We will start with a simple linear chain model

$$ \Large \varnothing \xrightarrow{v_1} x \xrightarrow{v_2} y \xrightarrow{v_3} \varnothing $$

where we want to read out the steady-state rate of $v_3$ dependent on the fixed concentration of $x$, while ignoring the inner state of the model.  


$$ \Large  x \xrightarrow{} ... \xrightarrow{v_3}$$

Since we need to fix a `variable` as an `parameter`, we can use the `make_variable_static` method to do that.

In [ ]:
# Now "x" is a parameter
get_linear_chain_2v().make_variable_static("x").parameters

And we can already create a function to create a model, which will take our surrogate as an input.  

In [ ]:
def get_model_with_surrogate(surrogate: SurrogateProtocol) -> Model:
    model = Model()
    model.add_variables({"x": 1.0, "z": 0.0})

    # Adding the surrogate
    model.add_surrogate(
        "surrogate",
        surrogate,
        args=["x"],
        outputs=["v2"],
        stoichiometries={
            "v2": {"x": -1, "z": 1},
        },
    )

    # Note that besides the surrogate we haven't defined any other reaction!
    # We could have though
    return model

### Create data

The surrogates used in the following will all use the **steady-state** fluxes depending on the inputs.  

We can thus create the necessary training data usign `scan.steady_state`.  
Since this is usually a large amount of data, we recommend caching the results using `Cache`. 

In [ ]:
surrogate_features = pd.DataFrame({"x": np.geomspace(1e-12, 2.0, 21)})

surrogate_targets = scan.steady_state(
    get_linear_chain_2v().make_variable_static("x"),
    to_scan=surrogate_features,
).fluxes.loc[:, ["v3"]]

# It's always a good idea to check the inputs and outputs
fig, (ax1, ax2) = plot.two_axes(figsize=(6, 3), sharex=False)
_ = plot.violins(surrogate_features, ax=ax1)[1].set(
    title="Features", ylabel="Flux / a.u."
)
_ = plot.violins(surrogate_targets, ax=ax2)[1].set(
    title="Targets", ylabel="Flux / a.u."
)
plt.show()

### Polynomial surrogate

We can train our polynomial surrogate using `train_polynomial_surrogate`.  
By default this will train polynomials for the degrees `(1, 2, 3, 4, 5, 6, 7)`, but you can change that by using the `degrees` argument.  
The function returns the trained surrogate and the training information for the different polynomial degrees.  

> **Currently the polynomial surrogates are limited to a single feature and a single target**


In [ ]:
surrogate, info = surrogates.poly.train(
    surrogate_features["x"],
    surrogate_targets["v3"],
)

print("Model", surrogate.model, end="\n\n")
print(info["score"])

You can then insert the surrogate into the model using the function we defined earlier


In [ ]:
concs, fluxes = (
    Simulator(get_model_with_surrogate(surrogate))
    .simulate(10)
    .get_result()
    .unwrap_or_err()
)

fig, (ax1, ax2) = plot.two_axes(figsize=(8, 3))
plot.lines(concs, ax=ax1)
plot.lines(fluxes, ax=ax2)
ax1.set(xlabel="time / a.u.", ylabel="concentration / a.u.")
ax2.set(xlabel="time / a.u.", ylabel="flux / a.u.")
plt.show()

While polynomial regression can model nonlinear relationships between variables, it often struggles when the underlying relationship is more complex than a polynomial function.  
You will learn about using neural networks in the next section.  

### Neural network surrogate using PyTorch

Neural networks are designed to capture highly complex and nonlinear relationships.  
Through layers of neurons and activation functions, neural networks can learn intricate patterns that are not easily represented by e.g. a polynomial.  
They have the flexibility to approximate any continuous function, given sufficient depth and appropriate training.  

You can train a neural network surrogate based on the popular [PyTorch](https://pytorch.org/) library using `train_torch_surrogate`.  
That function takes the `features`, `targets` and the number of `epochs` as inputs for it's training.  

`train_torch_surrogate` returns the trained surrogate, as well as the training `loss`.  
It is always a good idea to check whether that training loss approaches 0.   

In [ ]:
surrogate, loss = surrogates.torch.train(
    features=surrogate_features,
    targets=surrogate_targets,
    batch_size=100,
    epochs=250,
)

ax = loss.plot(ax=plt.subplots(figsize=(4, 2.5))[1])
ax.set_ylim(0, None)
plt.show()

As before, you can then insert the surrogate into the model using the function we defined earlier


In [ ]:
concs, fluxes = (
    Simulator(get_model_with_surrogate(surrogate))
    .simulate(10)
    .get_result()
    .unwrap_or_err()
)

fig, (ax1, ax2) = plot.two_axes(figsize=(8, 3))
plot.lines(concs, ax=ax1)
plot.lines(fluxes, ax=ax2)
ax1.set(xlabel="time / a.u.", ylabel="concentration / a.u.")
ax2.set(xlabel="time / a.u.", ylabel="flux / a.u.")
plt.show()

### Re-entrant training

Quite often you don't know the amount of epochs you are going to need in order to reach the required loss.  
In this case, you can directly use the `TorchSurrogateTrainer` class to continue training.  

In [ ]:
trainer = surrogates.torch.Trainer(
    features=surrogate_features,
    targets=surrogate_targets,
)

# First training epochs
trainer.train(epochs=100)
trainer.get_loss().plot(figsize=(4, 2.5)).set_ylim(0, None)
plt.show()

# Decide to continue training
trainer.train(epochs=150)
trainer.get_loss().plot(figsize=(4, 2.5)).set_ylim(0, None)
plt.show()

surrogate = trainer.get_surrogate(surrogate_outputs=["x"])

### Troubleshooting

It often can make sense to check specific predictions of the surrogate.  
For example, what does it predict when the inputs are all 0?  


In [ ]:
print(surrogate.predict_raw(np.array([-0.1])))
print(surrogate.predict_raw(np.array([0.0])))
print(surrogate.predict_raw(np.array([0.1])))

## Using keras instead of torch

If you installed keras, you can use it with exactly the same interface torch


In [ ]:
surrogate, loss = surrogates.keras.train(
    features=surrogate_features,
    targets=surrogate_targets,
    batch_size=100,
    epochs=250,
)

ax = loss.plot(ax=plt.subplots(figsize=(4, 2.5))[1])
ax.set_ylim(0, None)
plt.show()

## Using equinox instead of torch

In [ ]:
surrogate, loss = surrogates.equinox.train(
    features=surrogate_features,
    targets=surrogate_targets,
    batch_size=100,
    epochs=250,
    optimizer=optax.adamw(learning_rate=0.001),
)

ax = loss.plot(ax=plt.subplots(figsize=(4, 2.5))[1])
ax.set_ylim(0, None)
plt.show()

## Saving and loading surrogates

Training can be expensive, so a trained surrogate can be saved with `surrogate.save(path)` and restored with `surrogates.load(path)`.  
Persistence is delegated to the backend's own format.  
`path` is a directory holding `surrogate.json` (args, outputs, stoichiometries, units / targets) next to the model file:

| backend   | model file    | needs `model=` on load |
| --------- | ------------- | ---------------------- |
| `torch`   | `model.pt` (`state_dict`) | yes |
| `equinox` | `model.eqx` (`eqx.tree_serialise_leaves`) | yes |
| `keras`   | `model.keras` | no  |
| `poly`    | `model.json`  | no  |
| `qss`     | `model.dill`  | no  |

torch and equinox only store the weights, so loading needs a freshly built network with the same architecture (`model=`), into which the weights are loaded.  
A small function building your architecture makes that easy, and it also works for networks that `mxlpy.save` can't export (e.g. LSTMs or custom modules).

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory


def make_net() -> torch.nn.Module:
    return torch.nn.Sequential(
        torch.nn.Linear(1, 50),
        torch.nn.ReLU(),
        torch.nn.Linear(50, 50),
        torch.nn.ReLU(),
        torch.nn.Linear(50, 1),
    )


surrogate, _ = surrogates.torch.train(
    features=surrogate_features,
    targets=surrogate_targets,
    batch_size=100,
    epochs=250,
    model=make_net(),
)

with TemporaryDirectory() as tmp:
    surrogate.save(Path(tmp) / "v3")
    loaded = surrogates.load(Path(tmp) / "v3", model=make_net())

print(surrogate.predict_raw(np.array([0.5])))
print(loaded.predict_raw(np.array([0.5])))

`.keras` files store the architecture as well, so keras surrogates (and `poly` / `qss` ones) load without a template.

In [ ]:
import keras

surrogate, _ = surrogates.keras.train(
    features=surrogate_features,
    targets=surrogate_targets,
    batch_size=100,
    epochs=50,
    optimizer=keras.optimizers.Adam(),
)

with TemporaryDirectory() as tmp:
    surrogate.save(tmp)
    loaded = surrogates.load(tmp)

print(surrogate.predict_raw(np.array([0.5])))
print(loaded.predict_raw(np.array([0.5])))

Notes

- `save` overwrites an existing `surrogate.json` / model file in `path`.
- `qss` surrogates are pickled with `dill`: like any pickle, only load files you trust.
- This is independent of `mxlpy.save`, which exports a whole model including dense neural networks to the portable, cross-language `.mxl.json` format.

<div style="color: #ffffff; background-color: #04AA6D; padding: 3rem 1rem 3rem 1rem; box-sizing: border-box">
    <h2>First finish line</h2>
    With that you now know most of what you will need from a day-to-day basis about labelled models in mxlpy.
    <br />
    <br />
    Congratulations!
</div>

## Custom loss function

You can use a custom loss function by simply injecting a function that takes the predicted tensor `x` and the data `y` and produces another tensor.  

In [ ]:
import torch


def mean_abs(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return torch.mean(torch.abs(x - y))


trainer = surrogates.torch.Trainer(
    features=surrogate_features,
    targets=surrogate_targets,
    loss_fn=mean_abs,
)